In [1]:
import pandas as pd

df = pd.read_csv(r'C:\Users\huber\OneDrive\Pulpit\paysim dataset.csv')
print(df.head())

   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703             0.0             0.0        0               0  


In [2]:
df.shape

(6362620, 11)

In [3]:
df.columns

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud'],
      dtype='str')

In [4]:
df['isFraud'].value_counts()

isFraud
0    6354407
1       8213
Name: count, dtype: int64

In [5]:
df.groupby('isFraud')['amount'].mean()

isFraud
0    1.781970e+05
1    1.467967e+06
Name: amount, dtype: float64

In [11]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
df.to_sql('transactions', conn, if_exists='replace', index=False)

query = """
SELECT 
    isFraud, 
    AVG(oldbalanceOrg) as avg_old_balance,
    AVG(newbalanceOrig) as avg_new_balance
FROM transactions
GROUP BY isFraud
"""

result = pd.read_sql_query(query, conn)
print(result)

   isFraud  avg_old_balance  avg_new_balance
0        0     8.328287e+05    855970.228109
1        1     1.649668e+06    192392.631836


In [12]:
# SQL rule: Znalezienie kont, które wykonały więcej niż 3 transakcje w tym samym kroku czasowym
query_structuring = """
SELECT 
    nameOrig, 
    COUNT(*) as transaction_count,
    AVG(amount) as avg_amount
FROM transactions
WHERE isFraud = 1
GROUP BY nameOrig
HAVING transaction_count > 1
ORDER BY transaction_count DESC
"""

result_structuring = pd.read_sql_query(query_structuring, conn)
print(result_structuring.head(10))

Empty DataFrame
Columns: [nameOrig, transaction_count, avg_amount]
Index: []


In [13]:
query_flags = """
SELECT 
    type, 
    amount, 
    isFraud 
FROM transactions 
WHERE amount > 1000000
ORDER BY amount DESC
LIMIT 10
"""

result_flags = pd.read_sql_query(query_flags, conn)
print(result_flags)

       type       amount  isFraud
0  TRANSFER  92445516.64        0
1  TRANSFER  73823490.36        0
2  TRANSFER  71172480.42        0
3  TRANSFER  69886731.30        0
4  TRANSFER  69337316.27        0
5  TRANSFER  67500761.29        0
6  TRANSFER  66761272.21        0
7  TRANSFER  64234448.19        0
8  TRANSFER  63847992.58        0
9  TRANSFER  63294839.63        0


In [14]:
import pandas as pd
import sqlite3

df.to_csv('aml_analysis_report.csv', index=False)

readme_content = """
# AML Transaction Monitoring Prototype

## Problem Statement
Detecting financial fraud in a large-scale dataset (6.3M transactions) by analyzing behavioral patterns.

## Key Findings
- Fraud only occurs in TRANSFER and CASH_OUT types.
- Fraudulent transactions consistently result in a complete depletion of the sender's balance.
- High-value transactions (> 1M) are not necessarily fraudulent, proving that simple limit-based rules create high false-positive rates.

## Skills Demonstrated
- SQL: Data manipulation, aggregation, and hypothesis testing.
- Python (Pandas): Exploratory Data Analysis (EDA) and cleaning.
- Analytical Thinking: Identifying why high transaction amounts can be misleading.
"""

print(readme_content)


# AML Transaction Monitoring Prototype

## Problem Statement
Detecting financial fraud in a large-scale dataset (6.3M transactions) by analyzing behavioral patterns.

## Key Findings
- Fraud only occurs in TRANSFER and CASH_OUT types.
- Fraudulent transactions consistently result in a complete depletion of the sender's balance.
- High-value transactions (> 1M) are not necessarily fraudulent, proving that simple limit-based rules create high false-positive rates.

## Skills Demonstrated
- SQL: Data manipulation, aggregation, and hypothesis testing.
- Python (Pandas): Exploratory Data Analysis (EDA) and cleaning.
- Analytical Thinking: Identifying why high transaction amounts can be misleading.

